# 🎙️ OmniVoice — Tạo giọng nói (khách)

Chạy server voice của bạn trên GPU miễn phí của Colab. Chỉ cần **key bản quyền**.

**Cách dùng:** điền key vào ô 1 → menu **Runtime → Run all** → đợi ô 3 in ra
**API URL** và **API key**. Dùng cặp đó gọi tạo giọng từ máy bạn.

> Để **ô 3 chạy suốt** trong lúc dùng — tắt ô là tắt server. Mất mạng thì chạy
> lại ô 3, lấy URL mới.


In [ ]:
#@title 1. Nhập key bản quyền { display-mode: "form" }
#@markdown Dán key bản quyền của bạn rồi chạy ô này.
LICENSE_KEY = ""  #@param {type:"string"}

import hashlib, os, subprocess
LICENSE_KEY = LICENSE_KEY.strip()
assert LICENSE_KEY and not LICENSE_KEY.startswith("XXXX"), "Chưa nhập key bản quyền."

# device_id ổn định theo phiên máy Colab: hash của thông tin phần cứng có được.
def _fingerprint():
    bits = []
    for cmd in ("cat /proc/cpuinfo", "nvidia-smi -L", "cat /etc/machine-id"):
        try:
            bits.append(subprocess.check_output(cmd, shell=True, text=True,
                        stderr=subprocess.DEVNULL))
        except Exception:
            pass
    raw = ("|".join(bits) or os.uname().nodename).encode()
    return "colab-" + hashlib.sha256(raw).hexdigest()[:24]

DEVICE_ID = _fingerprint()
WORKERS = 4  #@param {type:"integer"}
print("Key nhận rồi. device_id =", DEVICE_ID)


In [ ]:
# ── 2. Cài thư viện + tải loader bảo mật ──────────────────────────────
import hashlib, urllib.request, importlib.util, sys, subprocess

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "httpx", "cryptography"], check=True)

BASE = "https://github.com/GrayPham/submodulevoice.git".replace(".git", "")
SO_URL  = f"{BASE}/releases/download/loader-linux/voice_loader-cp313-linux-x86_64.so"
SHA_URL = SO_URL + ".sha256"
SO_PATH = "/content/voice_loader-cp313-linux-x86_64.so"

print("Tải loader:", SO_URL)
urllib.request.urlretrieve(SO_URL, SO_PATH)
try:
    want = urllib.request.urlopen(SHA_URL, timeout=20).read().decode().split()[0]
    got = hashlib.sha256(open(SO_PATH, "rb").read()).hexdigest()
    assert got == want, f"checksum loader KHÔNG khớp:\n  cần {want}\n  được {got}"
    print("checksum loader khớp.")
except Exception as e:
    print("(bỏ qua kiểm checksum:", e, ")")

# Nạp loader .so vào như module 'voice_loader'.
spec = importlib.util.spec_from_file_location("voice_loader", SO_PATH)
voice_loader = importlib.util.module_from_spec(spec)
spec.loader.exec_module(voice_loader)
print("Loader nạp OK — phiên bản binary, kill-switch bên trong.")


In [ ]:
# ── 3. Đăng nhập + bật server (ĐỂ Ô NÀY CHẠY LIÊN TỤC) ────────────────
# Loader tự: xác thực key -> tải gói mã hoá từ R2 -> giải mã trong RAM ->
# bật server voice -> mở đường hầm -> in API URL + key. Định kỳ gọi về; nếu
# key bị thu hồi thì tự tắt.
rc = voice_loader.run(LICENSE_KEY, DEVICE_ID, WORKERS)
print("Kết thúc, rc =", rc)
